<a href="https://colab.research.google.com/github/MengOonLee/LLM/blob/main/References/LangChain/LangChainAcademy/LangChain/Foundation/CreateAgent/ipynb/1.5_personal_chef.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%bash
pip install --no-cache-dir -qU \
    langchain langgraph langchain-core \
    langchain-community langchain-openai \
    tavily-python

In [1]:
import warnings
warnings.filterwarnings('ignore')
import dotenv

_ = dotenv.load_dotenv(dotenv_path=".env", override=True)

In [ ]:
import os
import tavily
import typing
from langchain import chat_models, tools

llm = chat_models.init_chat_model(
    model="gpt-oss:120b-cloud",
    model_provider="openai",
    base_url="https://ollama.com/v1",
    api_key=os.environ["OLLAMA_API_KEY"],
    temperature=0
)

tavily_client = tavily.TavilyClient()
@tools.tool
def web_search(query: str) -> typing.Dict[str, typing.Any]:

    """Search the web for information"""

    return tavily_client.search(query=query)

In [ ]:
system_prompt = """
You are a personal chef. The user will give you a list of ingredients
they have left over in their house.

Using the web search tool, search the web for recipes that can be made
with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions
to the user, if requested.
"""

In [ ]:
from langchain import chat_models
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

In [ ]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="I have some leftover chicken and rice. What can I make?")]},
    config
)

print(response['messages'][-1].content)

In [ ]:
from pprint import pprint

pprint(response)